In [4]:
import time
import requests
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup

# pip install openpyxl # 이걸 해야 시트로 해서저장이 됨

In [ ]:
url = "https://finance.naver.com/"

driver = webdriver.Chrome()
driver.maximize_window()

driver.get(url)

time.sleep(2)

# action = ActionChains(driver)

domestic = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#menu > ul > li.m2 > a > span"
        ))
    )


domestic.click()  # popularItemList


# top10_finential = driver.find_elements(
#         By.ID,
#         "popularItemList"
#     )

req = driver.page_source

soup = BeautifulSoup(req, "html.parser")

top10_finential = soup.find("ul", "lst_pop").find_all("li")

fi_top_list = []
fi_name_list = []
fi_price_list = []

for row in top10_finential:
    fi_top = row.find("em").text
    fi_name = row.find("a").text
    fi_price = row.find("span").text

    fi_top_list.append(fi_top)
    fi_name_list.append(fi_name)
    fi_price_list.append(fi_price)

    #print(f"{fi_name}, {fi_price}")

df01 = pd.DataFrame({
    "순위": fi_top_list,
    '이름': fi_name_list,
    '가격': fi_price_list
})


international = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((
        By.CSS_SELECTOR, "#menu > ul > li.m3 > a > span"
    ))
)


international.click()

req = driver.page_source
soup = BeautifulSoup(req, "html.parser")



# 일단 스크롤 넣어보고
scroll_area = driver.find_element(By.XPATH,"/html/body")
before = 0

while True:
    driver.execute_script(
        "arguments[0].scrollTop = arguments[0].scrollHeight",
        scroll_area
    )

    time.sleep(1)

    after = driver.execute_script(
        "return arguments[0].scrollHeight",
        scroll_area
    )
    if before == after:
        break

    before = after


time.sleep(3)

important_international = driver.find_elements(
    By.XPATH,
    '//*[@id="americaIndex"]/thead/tr[position()>1]'
)

# important_international = driver.find_elements(By.CSS_SELECTOR, "#americaIndex > thead") # soup.find("tbody").find_all("tr") # soup.find('tb_status', 'thead').find_all('tr')  # driver.find_elements(By.CSS_SELECTOR, "#americaIndex tbody tr")
#americaIndex > thead > tr:nth-child(2)
# print(driver.page_source)
# print(soup.select("thead"))

# international_nation_list = []
# international_name_list = []
# international_cur_price_list = []
# international_percent_list = []
# international_time_list = []

# for row in important_international:
#     international_nation = row.find("td", class_="tb_td").text  td.tb_td
#     international_name = row.find("td", class_= "tb_td2").text    td.tb_td2
#     international_cur_price = row.find("td", class_="td.tb_td3").text
#     international_percent = row.find("td", class_="td.tb_td5").text
#     international_time = row.select_one("td", class_="td.tb_td7 span").text

#     international_nation_list.append(international_nation)
#     international_name_list.append(international_name)
#     international_cur_price_list.append(international_cur_price)
#     international_percent_list.append(international_percent)
#     international_time_list.append(international_time)

# driver.quit()

# # df.to_excel("fiential_top10.xlsx", index=False) # 기본 구조

# df02 = pd.DataFrame({
#     "국가": international_nation_list,
#     '지수명': international_name_list,
#     '현재가': international_cur_price_list,
#     "등락률": international_percent_list,
#     "시간": international_time_list

# })


important_international = driver.find_elements(By.CSS_SELECTOR, "#americaIndex thead tr")

international_nation_list = []
international_name_list = []
international_cur_price_list = []
international_percent_list = []
international_time_list = []

for row in important_international:
    # 2. 제목 행(국가, 지수명 등)을 건너뛰기 위한 방어 코드
    # 해당 tr 안에 td 태그가 없다면 데이터가 아니므로 패스합니다.
    tds = row.find_elements(By.CSS_SELECTOR, "td")
    if len(tds) == 0:
        continue
        
    try:
        # 3. 작성하셨던 클래스명을 기반으로 데이터를 추출합니다.
        # (만약 span 태그 관련 에러가 난다면 CSS_SELECTOR에서 ' span'을 지우고 실행해 보세요)
        international_nation = row.find_element(By.CSS_SELECTOR, "td.tb_td").text.strip()
        international_name = row.find_element(By.CSS_SELECTOR, "td.tb_td2").text.strip()
        international_cur_price = row.find_element(By.CSS_SELECTOR, "td.tb_td3").text.strip()
        international_percent = row.find_element(By.CSS_SELECTOR, "td.tb_td5").text.strip()
        international_time = row.find_element(By.CSS_SELECTOR, "td.tb_td7").text.strip()

        international_nation_list.append(international_nation)
        international_name_list.append(international_name)
        international_cur_price_list.append(international_cur_price)
        international_percent_list.append(international_percent)
        international_time_list.append(international_time)
        
    except Exception as e:
        # 구조가 다른 빈 줄이 있을 경우 에러 로그를 띄우고 계속 진행합니다.
        print(f"데이터 추출 건너뜀: {e}")
        continue

# 4. 데이터 프레임 생성
df02 = pd.DataFrame({
    "국가": international_nation_list,
    "지수명": international_name_list,
    "현재가": international_cur_price_list,
    "등락률": international_percent_list,
    "시간": international_time_list
})



# 이제 배민 추가해봄










#  df.to_excel("fiential_top10.xlsx", index=False)


with pd.ExcelWriter("finential.xlsx") as writer:
    df01.to_excel(writer, sheet_name="국내증시 탑10", index=False)
    df02.to_excel(writer, sheet_name="해외 주요 증시", index=False)

AttributeError: 'Worksheet' object has no attribute 'autofit'

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6e9fa3fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff6e9fa4000+14980]
	chromedriver!(No symbol) [0x7ff6e9ae7726]
	chromedriver!(No symbol) [0x7ff6e9b33083]
	chromedriver!(No symbol) [0x7ff6e9b6a2d2]
	chromedriver!(No symbol) [0x7ff6e9b640d2]
	chromedriver!(No symbol) [0x7ff6e9b63892]
	chromedriver!(No symbol) [0x7ff6e9ab03e5]
	chromedriver!GetHandleVerifier [0x7ff6ea587591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff6ea581902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff6ea5a7115+617a95]
	chromedriver!GetHandleVerifier [0x7ff6e9fc1dce+3274e]
	chromedriver!GetHandleVerifier [0x7ff6e9fca82c+3b1ac]
	chromedriver!(No symbol) [0x7ff6e9aaf188]
	chromedriver!GetHandleVerifier [0x7ff6ea757b58+7c84d8]
	KERNEL32!BaseThreadInitThunk [0x7ff92a737